# SNA Experiment 2 – Working with APIs
**Runtime:** Runtime → Change runtime type → **R**. Then Runtime → Run all.

The original task fetched data live from the **Twitter (X) API**. That API is now paid, so:
- Part 0 shows how an API call works using a **free public API** (JSONPlaceholder).
- Parts (a)–(d) answer the questions from the **already-collected** Twitter data in `senator_tweets.RData`.
- The Twitter calls we *would* make (R package `rtweet`) are shown as reference at the end.

In [ ]:
library(dplyr)
library(httr)
library(jsonlite)

## Part 0: what an API call looks like
A **GET** request to a URL → the server answers with **JSON** → we convert it to a data frame.
Status code **200** = OK.

In [ ]:
response <- GET("https://jsonplaceholder.typicode.com/users")
cat("Status code:", status_code(response), "\n")

users <- fromJSON(content(response, "text", encoding = "UTF-8"))
users %>% select(id, name, username, email)

## Load the data and link Twitter handles to senators

In [ ]:
load(url("https://dssoc.github.io/datasets/congress.RData"))
load(url("https://dssoc.github.io/datasets/senator_tweets.RData"))

# senators with their Twitter handle (handles stored in lowercase so they match)
senators <- congress %>%
  filter(type == "sen") %>%
  inner_join(congress_contact, by = "bioguide_id") %>%
  mutate(twitter = tolower(twitter))

tweets <- senator_tweet_sample %>%
  mutate(screen_name = tolower(screen_name))

# keep only tweets whose account belongs to a senator in our dataset; add gender, age
tweets <- tweets %>%
  inner_join(senators %>% select(twitter, full_name, gender, birthyear),
             by = c("screen_name" = "twitter"))

head(tweets %>% select(full_name, gender, created_at, retweet_count))

## (a) How many senator Twitter accounts listed in our dataset were found on Twitter?

In [ ]:
listed <- sum(!is.na(senators$twitter) & senators$twitter != "")
found  <- n_distinct(tweets$screen_name)

cat("Senators with a Twitter handle listed:", listed, "\n")
cat("Of those, accounts found (have tweets in the data):", found, "\n")

## (b) Average number of published Tweets per senator, by gender

In [ ]:
tweets_per_senator <- tweets %>%
  count(screen_name, gender, name = "n_tweets")

tweets_per_senator %>%
  group_by(gender) %>%
  summarise(avg_tweets = mean(n_tweets), senators = n())

## (c) Correlation between a senator's age and when their Twitter account was created
The dataset has no account-creation date (the API field `account_created_at`).
**Approximation:** the date of the account's **earliest tweet in our data** (the account must exist before its first tweet).

In [ ]:
first_tweet <- tweets %>%
  group_by(screen_name, birthyear) %>%
  summarise(first_seen = min(created_at), .groups = "drop") %>%
  mutate(age = 2022 - birthyear)

r <- cor(first_tweet$age, as.numeric(first_tweet$first_seen))
cat("Correlation (age vs first tweet date):", round(r, 3), "\n")

## (d) Last two Tweets of each senator → average retweets by gender

In [ ]:
last_two <- tweets %>%
  group_by(screen_name) %>%
  arrange(desc(created_at)) %>%
  slice_head(n = 2) %>%
  ungroup()

cat("Tweets selected:", nrow(last_two), "\n")

last_two %>%
  group_by(gender) %>%
  summarise(avg_retweets = mean(retweet_count), tweets = n())

## Reference: the same questions with the real Twitter API (needs a paid developer token, not run)
```r
library(rtweet)
auth_setup_default()                                  # login / bearer token
users  <- lookup_users(senators$twitter)               # (a) found accounts = nrow(users)
users$statuses_count                                   # (b) total published tweets
users$account_created_at                               # (c) real creation date
tl     <- get_timeline(senators$twitter, n = 2)        # (d) last two tweets each
```